# Leakage-safe 12-lead ECG classification

This notebook is the reproducible classical experiment entry point. The original exploratory notebook is retained for provenance, but its lead-level random split and dataframe slicing must not be used for scientific comparisons.

The protocol keeps each original ECG intact, preserves co-occurring diagnoses as multilabel targets, creates grouped label-balanced splits, fits normalization on training data only, selects checkpoints using validation macro-AUPRC, and opens the test set once after training.

In [ ]:
from collections import Counter
from pathlib import Path
import json
import subprocess
import sys

from ecg_experiment.config import DEFAULT_TARGET_CLASSES, resolve_repo_root
from ecg_experiment.data import build_manifest, make_grouped_splits

repo_root = resolve_repo_root()
repo_root

## Build one-record-per-ECG manifest

Each original record appears once. Co-occurring selected diagnoses become a multi-hot target trained with `BCEWithLogitsLoss`; records are never copied into conflicting class examples.

In [ ]:
records = build_manifest(
    repo_root / 'Data' / 'WFDBRecords',
    repo_root / 'References' / 'ConditionNames_SNOMED-CT.csv',
    DEFAULT_TARGET_CLASSES,
    require_single_target=False,
)
print('Eligible records:', len(records))
Counter(label for record in records for label in record.labels)

## Create and audit fixed splits

Do not change these folds after inspecting test results. Model and hyperparameter choices belong to the training and validation splits only.

In [ ]:
splits = make_grouped_splits(
    records, n_splits=5, test_fold=0, validation_fold=0, seed=43,
    multilabel=True, class_names=DEFAULT_TARGET_CLASSES,
)
for name, split in [('train', splits.train), ('validation', splits.validation), ('test', splits.test)]:
    print(name, len(split), Counter(label for record in split for label in record.labels))

train_ids = {record.record_id for record in splits.train}
validation_ids = {record.record_id for record in splits.validation}
test_ids = {record.record_id for record in splits.test}
assert not (train_ids & validation_ids or train_ids & test_ids or validation_ids & test_ids)
print('Leakage audit passed: record sets are disjoint.')

## Run the prespecified classical baseline

The command writes its configuration, exact split manifest, training-only normalization statistics, best checkpoint, learning history, and one final test evaluation under `artifacts/classical/`. Use a new output directory for every seed.

In [ ]:
command = [
    sys.executable,
    str(repo_root / 'Code' / 'run_classical_experiment.py'),
    '--repo-root', str(repo_root),
    '--output-dir', 'artifacts/classical/seed-43',
    '--seed', '43', '--stage', 'validate',
    '--epochs', '100',
    '--patience', '12',
]
subprocess.run(command, check=True)

## Read the sealed test result

Run this only after the architecture and training choices are final. For a publication-quality estimate, repeat the complete fixed protocol across prespecified seeds and report confidence intervals rather than selecting the best seed.

In [ ]:
RUN_SEALED_TEST = False
if RUN_SEALED_TEST:
    test_command = command.copy()
    test_command[test_command.index('--stage') + 1] = 'test'
    subprocess.run(test_command, check=True)
else:
    print('Test set remains sealed.')